# 11. 의약품개요정보 (e약은요) EDA (T0-2)

**데이터 출처**: data.go.kr OpenAPI  
**파이프라인 역할**: Stage 3 보고서 생성의 텍스트 원천

## EDA 체크리스트
- [ ] 총 품목 수, 필드별 채움률
- [ ] 주요 필드 토큰 길이 분포 (GPT-4o 컨텍스트 예산 확인)
- [ ] 상호작용 필드 구조 분석 (코드 참조 vs. 자유텍스트)
- [ ] 낱알식별(T0-1)과 품목일련번호 join 히트율
- [ ] 인코딩 확인 (HTML 태그 잔재 등)
- [ ] 산출물: `data/interim/eyak_clean.parquet` + `eyak_summary.json`

In [ ]:
import os, json, time, re
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
from pathlib import Path
from tqdm import tqdm

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../../.env").resolve())
except ImportError:
    pass

ROOT = Path("../../").resolve()
RAW = ROOT / "data" / "raw" / "eyak"
INTERIM = ROOT / "data" / "interim"
RAW.mkdir(parents=True, exist_ok=True)

DATAGOKR_KEY = os.environ.get("DATAGOKR_EYAK_KEY", "")
API_URL = "http://apis.data.go.kr/1471000/DrbEasyDrugInfoService/getDrbEasyDrugList"

In [ ]:
def fetch_page(page, rows=100):
    params = {
        "serviceKey": DATAGOKR_KEY,
        "pageNo": page,
        "numOfRows": rows,
        "type": "json"
    }
    resp = requests.get(API_URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

first = fetch_page(1, rows=1)
total_count = first.get("body", {}).get("totalCount", 0)
print(f"총 {total_count:,}건")

In [ ]:
cache_file = RAW / "eyak_all.json"

if cache_file.exists():
    with open(cache_file, encoding="utf-8") as f:
        all_items = json.load(f)
    print(f"캐시 로드: {len(all_items):,}건")
else:
    ROWS = 100
    pages = (total_count + ROWS - 1) // ROWS
    all_items = []
    for p in tqdm(range(1, pages + 1)):
        data = fetch_page(p, rows=ROWS)
        items = data.get("body", {}).get("items", [])
        all_items.extend(items)
        time.sleep(0.1)
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(all_items, f, ensure_ascii=False)
    print(f"수집 완료: {len(all_items):,}건")

In [ ]:
df = pd.DataFrame(all_items)
print(f"shape: {df.shape}")
print(df.columns.tolist())

In [ ]:
# 필드별 채움률
target_fields = ["efcyQesitm", "useMethodQesitm", "atpnWarnQesitm", "atpnQesitm",
                 "intrcQesitm", "seQesitm", "depositMethodQesitm"]
fill_rate = {}
for col in target_fields:
    if col in df.columns:
        rate = (df[col].notna() & (df[col].str.strip() != "")).mean()
        fill_rate[col] = f"{rate:.1%}"
print(pd.Series(fill_rate).to_string())

In [ ]:
# 토큰 길이 분포 (한국어 기준 공백 split 기반 rough estimate)
def rough_token_count(text):
    if not isinstance(text, str):
        return 0
    # 한글 1자 ≈ 1.5 토큰, 영어 1단어 ≈ 1.3 토큰 (rough)
    return int(len(text) / 1.5)

fig, axes = plt.subplots(len(target_fields), 1, figsize=(10, 3 * len(target_fields)))
for ax, col in zip(axes, target_fields):
    if col in df.columns:
        lens = df[col].dropna().apply(rough_token_count)
        lens.hist(bins=50, ax=ax)
        ax.set_title(f"{col} — 중앙값 {lens.median():.0f} / p95 {lens.quantile(0.95):.0f} 토큰")
        ax.set_xlabel("rough token count")
plt.tight_layout()
plt.show()

In [ ]:
# 10종 약물 동시 조회 시 최악 시나리오 토큰 추정
if "efcyQesitm" in df.columns:
    worst_10 = df["efcyQesitm"].dropna().apply(rough_token_count).nlargest(10).sum()
    print(f"효능효과 최대 10종 합산 rough 토큰: {worst_10:,} (GPT-4o 컨텍스트 128k — 여유: {128000 - worst_10:,})")

In [ ]:
# 상호작용 필드 구조 분석 — 코드 참조 패턴 탐지
if "intrcQesitm" in df.columns:
    sample_intrc = df["intrcQesitm"].dropna().head(20)
    for i, text in enumerate(sample_intrc[:5]):
        print(f"[{i}]", text[:300], "\n")

In [ ]:
# HTML 태그 잔재 확인
html_pattern = re.compile(r'<[^>]+>')
for col in target_fields:
    if col in df.columns:
        has_html = df[col].dropna().apply(lambda x: bool(html_pattern.search(x))).sum()
        if has_html:
            print(f"{col}: HTML 태그 포함 {has_html:,}건")

In [ ]:
# 낱알식별과 join 히트율
nadal_path = INTERIM / "nadal_ident_clean.parquet"
if nadal_path.exists():
    df_nadal = pd.read_parquet(nadal_path, columns=["ITEM_SEQ"])
    id_col = "itemSeq" if "itemSeq" in df.columns else None
    if id_col:
        merged = df[id_col].astype(str).isin(df_nadal["ITEM_SEQ"].astype(str))
        print(f"e약은요 → 낱알식별 join 히트율: {merged.mean():.1%} ({merged.sum():,}/{len(df):,})")
    else:
        print("품목일련번호 컬럼명 확인 필요:", df.columns.tolist())
else:
    print("낱알식별 parquet 없음 — 10_nadal_ident 먼저 실행 필요")

In [ ]:
# 산출물 저장
df.to_parquet(INTERIM / "eyak_clean.parquet", index=False)

summary = {
    "total_rows": len(df),
    "columns": df.columns.tolist(),
    "fill_rates": fill_rate,
}
with open(INTERIM / "eyak_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("저장 완료")